# Fire Extinguisher Detection - YOLOv11 Training

**Project 1:** Object Detection and Localization  
**Model:** YOLOv11  
**Author:** Kelechi Ezekwe (@kelechiezekwe)  
**Platform:** Google Colab (Free T4 GPU)

---

## Overview
Train YOLOv11 to detect fire extinguishers from the Roboflow dataset.

**Performance Targets:**
- Real-object TPR ≥ 0.75
- Decoy FPR ≤ 0.15

---

## Setup Instructions

1. Open this notebook in Google Colab
2. Runtime → Change runtime type → T4 GPU
3. Run cells sequentially
4. Download trained weights at the end

## 1. Environment Setup

In [ ]:
# Check GPU availability
!nvidia-smi

import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"CUDA device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None'}")

In [ ]:
# Install dependencies
!pip install -q ultralytics roboflow

from ultralytics import YOLO
import os
from pathlib import Path
import yaml

## 2. Mount Google Drive (Optional - for saving weights)

In [ ]:
# Uncomment to mount Google Drive for persistent storage
# from google.colab import drive
# drive.mount('/content/drive')

# Create directory in Drive
# DRIVE_DIR = '/content/drive/MyDrive/CV_Project1'
# !mkdir -p {DRIVE_DIR}/weights {DRIVE_DIR}/results

## 3. Download Dataset from Roboflow

**Dataset:** Fire Extinguisher Detection  
**Link:** https://app.roboflow.com/kaycee-hn2a6/fireextinguisher-z5atr-6ktlv/train

In [ ]:
from roboflow import Roboflow

# Initialize Roboflow
# Get your API key from: https://app.roboflow.com/settings/api
API_KEY = "YOUR_ROBOFLOW_API_KEY_HERE"  # Replace with your actual key

rf = Roboflow(api_key=API_KEY)
project = rf.workspace("kaycee-hn2a6").project("fireextinguisher-z5atr-6ktlv")
dataset = project.version(1).download("yolov8")  # YOLOv8 format works for YOLOv11

print(f"Dataset downloaded to: {dataset.location}")

### Alternative: Manual Dataset Upload

If you exported dataset from Roboflow as ZIP:

In [ ]:
# Option 1: Upload ZIP file using Colab file browser
# from google.colab import files
# uploaded = files.upload()
# !unzip -q fireextinguisher.zip -d dataset/

# Option 2: Download from Google Drive
# !cp /content/drive/MyDrive/CV_Project1/dataset.zip .
# !unzip -q dataset.zip -d dataset/

## 4. Inspect Dataset

In [ ]:
# Check dataset structure
DATA_PATH = dataset.location if 'dataset' in locals() else 'dataset'

!ls -lh {DATA_PATH}
!ls {DATA_PATH}/train/images | head -5
!ls {DATA_PATH}/train/labels | head -5

In [ ]:
# Read data.yaml
import yaml

yaml_path = f"{DATA_PATH}/data.yaml"
with open(yaml_path, 'r') as f:
    data_config = yaml.safe_load(f)

print("Dataset Configuration:")
print(yaml.dump(data_config, default_flow_style=False))

# Count images
import glob
train_images = len(glob.glob(f"{DATA_PATH}/train/images/*"))
val_images = len(glob.glob(f"{DATA_PATH}/valid/images/*"))

print(f"\nDataset Statistics:")
print(f"  Training images: {train_images}")
print(f"  Validation images: {val_images}")
print(f"  Classes: {data_config.get('names', [])}")

## 5. Visualize Sample Images

In [ ]:
import matplotlib.pyplot as plt
import cv2
import random

def plot_sample_images(data_path, split='train', num_samples=4):
    """Plot random sample images with their bounding boxes"""
    img_dir = f"{data_path}/{split}/images"
    label_dir = f"{data_path}/{split}/labels"
    
    img_files = glob.glob(f"{img_dir}/*")
    samples = random.sample(img_files, min(num_samples, len(img_files)))
    
    fig, axes = plt.subplots(2, 2, figsize=(12, 10))
    axes = axes.flatten()
    
    for idx, img_path in enumerate(samples):
        # Read image
        img = cv2.imread(img_path)
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        h, w = img.shape[:2]
        
        # Read labels
        label_path = img_path.replace('/images/', '/labels/').replace('.jpg', '.txt').replace('.png', '.txt')
        
        if os.path.exists(label_path):
            with open(label_path, 'r') as f:
                labels = f.readlines()
            
            # Draw bboxes
            for label in labels:
                class_id, x_center, y_center, width, height = map(float, label.split())
                
                # Convert YOLO format to pixel coordinates
                x1 = int((x_center - width/2) * w)
                y1 = int((y_center - height/2) * h)
                x2 = int((x_center + width/2) * w)
                y2 = int((y_center + height/2) * h)
                
                cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                cv2.putText(img, f"Class {int(class_id)}", (x1, y1-10), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 2)
        
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(os.path.basename(img_path))
    
    plt.tight_layout()
    plt.savefig('dataset_samples.png', dpi=150, bbox_inches='tight')
    plt.show()

plot_sample_images(DATA_PATH, split='train', num_samples=4)

## 6. Initialize YOLOv11 Model

In [ ]:
# Load YOLOv11 nano model (fast training, good for starting)
# Options: yolo11n.pt (nano), yolo11s.pt (small), yolo11m.pt (medium)

MODEL_SIZE = 'n'  # n=nano, s=small, m=medium
model = YOLO(f'yolo11{MODEL_SIZE}.pt')

print(f"Model loaded: YOLOv11{MODEL_SIZE}")
print(f"Parameters: {sum(p.numel() for p in model.model.parameters())/1e6:.2f}M")

## 7. Training Configuration

In [ ]:
# Training hyperparameters
EPOCHS = 100          # Number of training epochs
BATCH_SIZE = 16       # Batch size (adjust based on GPU memory)
IMG_SIZE = 640        # Input image size
PATIENCE = 15         # Early stopping patience
PROJECT_NAME = 'fire_extinguisher_detection'
RUN_NAME = 'yolo11n_v1'

training_config = {
    'data': f"{DATA_PATH}/data.yaml",
    'epochs': EPOCHS,
    'batch': BATCH_SIZE,
    'imgsz': IMG_SIZE,
    'patience': PATIENCE,
    'project': PROJECT_NAME,
    'name': RUN_NAME,
    'save': True,
    'save_period': 10,  # Save checkpoint every 10 epochs
    'device': 0,  # GPU device
    'workers': 8,
    'optimizer': 'AdamW',
    'lr0': 0.001,  # Initial learning rate
    'lrf': 0.01,   # Final learning rate factor
    'momentum': 0.937,
    'weight_decay': 0.0005,
    'warmup_epochs': 3,
    'cos_lr': True,  # Cosine learning rate scheduler
    'augment': True,
    'hsv_h': 0.015,  # Hue augmentation
    'hsv_s': 0.7,    # Saturation
    'hsv_v': 0.4,    # Value
    'degrees': 10.0, # Rotation
    'translate': 0.1,
    'scale': 0.5,
    'shear': 2.0,
    'perspective': 0.0,
    'flipud': 0.0,   # Vertical flip probability
    'fliplr': 0.5,   # Horizontal flip probability
    'mosaic': 1.0,   # Mosaic augmentation
    'mixup': 0.1,    # Mixup augmentation
}

print("Training Configuration:")
for key, value in training_config.items():
    print(f"  {key}: {value}")

## 8. Train Model

In [ ]:
# Start training
print("Starting training...")
print(f"This will take approximately {EPOCHS * 2 / 60:.1f} hours on T4 GPU")

results = model.train(**training_config)

print("\nTraining complete!")

## 9. Evaluate Model Performance

In [ ]:
# Evaluate on validation set
metrics = model.val()

print("\n=== Validation Metrics ===")
print(f"mAP50: {metrics.box.map50:.4f}")
print(f"mAP50-95: {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall: {metrics.box.mr:.4f}")

In [ ]:
# Plot training curves
from IPython.display import Image, display

results_dir = f"{PROJECT_NAME}/{RUN_NAME}"

print("Training curves:")
display(Image(filename=f"{results_dir}/results.png"))

print("\nConfusion matrix:")
display(Image(filename=f"{results_dir}/confusion_matrix.png"))

## 10. Test Predictions on Sample Images

In [ ]:
# Load best weights
best_model = YOLO(f"{results_dir}/weights/best.pt")

# Test on validation images
test_images = glob.glob(f"{DATA_PATH}/valid/images/*")[:5]

for img_path in test_images:
    results = best_model.predict(img_path, conf=0.25, save=True, project='predictions')
    print(f"Predictions saved for: {os.path.basename(img_path)}")

In [ ]:
# Display prediction samples
pred_images = glob.glob('predictions/predict/*.jpg')

fig, axes = plt.subplots(1, min(3, len(pred_images)), figsize=(15, 5))
if len(pred_images) == 1:
    axes = [axes]

for ax, img_path in zip(axes, pred_images[:3]):
    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(os.path.basename(img_path))

plt.tight_layout()
plt.savefig('sample_predictions.png', dpi=150, bbox_inches='tight')
plt.show()

## 11. Export Model

In [ ]:
# Export to different formats (optional)
# best_model.export(format='onnx')  # ONNX format
# best_model.export(format='torchscript')  # TorchScript

print(f"Best model weights: {results_dir}/weights/best.pt")
print(f"Last model weights: {results_dir}/weights/last.pt")

## 12. Save Results and Download Weights

In [ ]:
# Create results summary
import json

summary = {
    'model': f'YOLOv11{MODEL_SIZE}',
    'epochs': EPOCHS,
    'batch_size': BATCH_SIZE,
    'img_size': IMG_SIZE,
    'training_images': train_images,
    'validation_images': val_images,
    'metrics': {
        'mAP50': float(metrics.box.map50),
        'mAP50-95': float(metrics.box.map),
        'precision': float(metrics.box.mp),
        'recall': float(metrics.box.mr),
    },
    'best_weights': f"{results_dir}/weights/best.pt",
}

with open('training_summary.json', 'w') as f:
    json.dump(summary, f, indent=2)

print("Training summary saved to: training_summary.json")
print(json.dumps(summary, indent=2))

In [ ]:
# Download weights and results to local machine
from google.colab import files

# Option 1: Download directly
print("Downloading best weights...")
files.download(f"{results_dir}/weights/best.pt")

print("Downloading training summary...")
files.download('training_summary.json')

print("Downloading training curves...")
files.download(f"{results_dir}/results.png")

# Option 2: Copy to Google Drive
# !cp {results_dir}/weights/best.pt /content/drive/MyDrive/CV_Project1/weights/
# !cp training_summary.json /content/drive/MyDrive/CV_Project1/results/

print("\n✓ Downloads complete!")

## 13. Next Steps

After training completes:

1. **Download Files:**
   - `best.pt` → Save to `models/weights/`
   - `training_summary.json` → Save to `results/`
   - `results.png` → Save to `results/figures/`

2. **Create Baseline Detector** (no decoy filter)
   - Use trained YOLO model
   - Accept all detections as real
   - Use median depth for 3D position

3. **Prepare Evaluation Dataset**
   - Label 127 RGB images from `raw/eval/camera_color_image_raw/`
   - Mark real fire extinguishers vs decoys
   - Create bounding box annotations

4. **Run Evaluation Script**
   - Compute TPR, FPR
   - Compare baseline vs final system
   - Generate visual evidence

5. **Create Presentation**
   - 8-12 slides
   - System architecture
   - Results and metrics
   - Reflections and limitations

---

**GitHub:** kelechiezekwe  
**Due:** October 30, 2026, 11:59pm